# 04 — RAG avec Embeddings + Mistral API

**Objectif :** améliorer la traduction médicale EN→FR via un système RAG :
- **Retrieval** : embeddings denses (`intfloat/multilingual-e5-base`) + FAISS
- **Generation** : `mistral-small-latest` via API Mistral
- **Évaluation** : MEDCON + BLEU + COMET sur les traductions RAG vs GPT-4o


## 0. Setup

In [ ]:
import subprocess, os
subprocess.run(['pip', 'install', '-q', '--upgrade', 'numpy'])
os.kill(os.getpid(), 9)

In [1]:
import os

if not os.path.exists('/content/Evaluating_medical_machine_translation'):
    !git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

Répertoire : /content/Evaluating_medical_machine_translation


In [2]:
from google.colab import files

os.makedirs('data', exist_ok=True)

# Uploader : corpus_WMT24.json + unique_contexts_for_RAG.json + cleaned_mesh_snomed_dico.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  → {dest}')

Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving corpus_WMT24.json to corpus_WMT24.json
Saving unique_contexts_for_RAG.json to unique_contexts_for_RAG.json
  → data/cleaned_mesh_snomed_dico.json
  → data/corpus_WMT24.json
  → data/unique_contexts_for_RAG.json


In [6]:
!pip uninstall mistralai -y
!pip install mistralai==1.2.5 --quiet
!pip install -q pyahocorasick sacrebleu sentence-transformers faiss-cpu transformers accelerate

Found existing installation: mistralai 2.4.2
Uninstalling mistralai-2.4.2:
  Successfully uninstalled mistralai-2.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.


## 1. Imports

In [1]:
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import torch
import sacrebleu
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import faiss
from mistralai import Mistral
from google.colab import userdata

mistral_client = Mistral(api_key=userdata.get('MISTRAL_API_KEY'))
print('Mistral OK')

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

Mistral OK
Imports OK


## 2. Chargement des données

In [2]:
with open('data/corpus_WMT24.json', encoding='utf-8') as f:
    test_docs = json.load(f)

with open('data/unique_contexts_for_RAG.json', encoding='utf-8') as f:
    rag_contexts = json.load(f)

with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict = json.load(f)

rag_texts = [str(c) for c in rag_contexts]

print(f'Corpus WMT24    : {len(test_docs)} documents')
print(f'Contextes RAG   : {len(rag_texts)} documents FR')
print(f'Dictionnaire    : {len(raw_dict):,} entrées')

Corpus WMT24    : 49 documents
Contextes RAG   : 10995 documents FR
Dictionnaire    : 2,515 entrées


## 3. Construction de l'index FAISS avec embeddings

In [3]:
print('Chargement du modèle d\'embeddings (multilingual-e5-base)...')
embed_model = SentenceTransformer('intfloat/multilingual-e5-base')

print('Encodage des contextes RAG (peut prendre quelques minutes)...')
# multilingual-e5 requiert le préfixe "passage: " pour les documents
rag_embeddings = embed_model.encode(
    [f'passage: {t}' for t in rag_texts],
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Index FAISS (cosine similarity via inner product sur vecteurs normalisés)
dim = rag_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(rag_embeddings.astype('float32'))

print(f'Index FAISS construit : {index.ntotal} vecteurs de dimension {dim}')


def retrieve_top_k(query_text, k=3):
    """Retourne les k contextes FR les plus proches de la requête EN."""
    query_emb = embed_model.encode(
        [f'query: {query_text}'],
        normalize_embeddings=True
    ).astype('float32')
    scores, indices = index.search(query_emb, k)
    return [rag_texts[i] for i in indices[0]], scores[0]


# Test rapide
ctx_test, scores_test = retrieve_top_k(test_docs[0]['text_en'], k=3)
print(f'\nTest retrieval (doc #0) — scores : {scores_test.round(3)}')
print(f'Contexte top-1 (extrait) : {ctx_test[0][:200]}...')

Chargement du modèle d'embeddings (multilingual-e5-base)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Encodage des contextes RAG (peut prendre quelques minutes)...


Batches:   0%|          | 0/172 [00:00<?, ?it/s]

Index FAISS construit : 10995 vecteurs de dimension 768

Test retrieval (doc #0) — scores : [0.819 0.819 0.818]
Contexte top-1 (extrait) : L’objectif général de cette étude était de montrer comment utiliser les renseignements recueillis dans le cadre du Programme de la sécurité des produits de consommation (le Programme) pour identifier ...


## 4. Initialisation client Mistral

In [4]:
mistral_client = Mistral(api_key=userdata.get('MISTRAL_API_KEY'))
MODEL_NAME = 'mistral-small-latest'
print(f'Client Mistral initialisé — modèle : {MODEL_NAME}')


def translate_with_rag(source_en, k=3):
    """Traduit source_en en FR via RAG + API Mistral."""
    contexts, _ = retrieve_top_k(source_en, k=k)
    context_block = '\n\n'.join([
        f'Example {i+1} of French medical text:\n{ctx}'
        for i, ctx in enumerate(contexts)
    ])
    prompt = f"""You are an expert medical translator. Translate the following English medical text into French.
To help you, here are examples of French medical texts from the same domain:

{context_block}

English text to translate:
{source_en}

Provide only the French translation, without any explanation or comment."""

    response = mistral_client.chat.complete(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1000
    )
    return response.choices[0].message.content.strip()


# Test rapide
print('\nTest traduction (doc #0)...')
test_trad = translate_with_rag(test_docs[0]['text_en'])
print(f'{test_trad[:300]}...')


Client Mistral initialisé — modèle : mistral-small-latest

Test traduction (doc #0)...
L’objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s’agissait d’une étude rétrospective et descriptive menée sur une période de 19 ans (2002-2020) et portant sur les décès par asphyxie mécanique pris en charge par la...


## 5. Génération des 49 traductions RAG

In [5]:
print('=' * 80)
print('GÉNÉRATION TRADUCTIONS RAG — Mistral API + Embeddings (k=3)')
print('=' * 80 + '\n')

rag_translations = []
for i, doc in enumerate(tqdm(test_docs, desc='RAG translate')):
    try:
        translation = translate_with_rag(doc['text_en'], k=3)
        rag_translations.append(translation)
        time.sleep(0.3)  # rate limiting
    except Exception as e:
        print(f'Erreur doc {i} : {e}')
        rag_translations.append('')

print(f'\n{len(rag_translations)} traductions générées.')
print(f'\nExemple doc #0 :\n{rag_translations[0][:400]}...')


GÉNÉRATION TRADUCTIONS RAG — Mistral API + Embeddings (k=3)



RAG translate:   0%|          | 0/49 [00:00<?, ?it/s]


49 traductions générées.

Exemple doc #0 :
L’objectif de ce travail était de décrire les décès par asphyxie mécanique survenus à Abidjan afin de contribuer à leur prévention. Il s’agissait d’une étude rétrospective et descriptive menée sur une période de 19 ans (2002-2020) et portant sur les décès par asphyxie mécanique pris en charge par la Médecine légale. Les décès par asphyxie mécanique représentaient 1,23 % (756/60 984), concernaient ...


## 6. MEDCON sur les traductions RAG

In [6]:
pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print('Calcul MEDCON — RAG...')
results_rag = []
for i, doc in enumerate(tqdm(test_docs, desc='MEDCON RAG')):
    r = medcon_grouped(
        doc['text_en'],
        rag_translations[i],
        pairs, automaton_en, automaton_fr
    )
    results_rag.append({
        'doc_id'           : i,
        'rag_fr'           : rag_translations[i],
        'medcon_f1'        : r['f1'],
        'medcon_precision' : r['precision'],
        'medcon_recall'    : r['recall'],
        'n_expected'       : r['n_expected'],
        'n_found'          : r['n_found'],
        'n_match'          : r['n_match'],
        'matched'          : r['matched'],
        'missed'           : r['missed'],
        'extra'            : r['extra'],
    })

df_rag = pd.DataFrame(results_rag)

print(f'\nMEDCON RAG :')
print(f'  Precision : {df_rag["medcon_precision"].mean():.3f}')
print(f'  Recall    : {df_rag["medcon_recall"].mean():.3f}')
print(f'  F1        : {df_rag["medcon_f1"].mean():.3f}')

Calcul MEDCON — RAG...


MEDCON RAG:   0%|          | 0/49 [00:00<?, ?it/s]


MEDCON RAG :
  Precision : 0.432
  Recall    : 0.408
  F1        : 0.414


## 7. BLEU sur les traductions RAG

In [7]:
print('Calcul BLEU — RAG...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU RAG')):
    if rag_translations[i]:
        bleu = sacrebleu.sentence_bleu(rag_translations[i], [doc['translation_fr']]).score
    else:
        bleu = 0.0
    df_rag.loc[i, 'bleu'] = bleu

print(f'BLEU moyen RAG : {df_rag["bleu"].mean():.2f}')

Calcul BLEU — RAG...


BLEU RAG:   0%|          | 0/49 [00:00<?, ?it/s]

BLEU moyen RAG : 49.58


## 8. COMET sur les traductions RAG

In [8]:
USE_COMET = False
try:
    from comet import download_model, load_from_checkpoint

    print('Chargement COMET (wmt22-comet-da)...')
    comet_model = load_from_checkpoint(download_model('Unbabel/wmt22-comet-da'))

    comet_data_rag = [
        {'src': doc['text_en'], 'mt': rag_translations[i], 'ref': doc['translation_fr']}
        for i, doc in enumerate(test_docs)
        if rag_translations[i]
    ]
    valid_idx = [i for i, doc in enumerate(test_docs) if rag_translations[i]]

    comet_output = comet_model.predict(
        comet_data_rag,
        batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for j, i in enumerate(valid_idx):
        df_rag.loc[i, 'comet'] = float(comet_output.scores[j])

    USE_COMET = True
    print(f'COMET moyen RAG : {df_rag["comet"].mean():.3f}')

except Exception as e:
    print(f'COMET indisponible : {e}')
    df_rag['comet'] = float('nan')


Chargement COMET (wmt22-comet-da)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

COMET indisponible : No module named 'entmax'


print('Calcul MEDCON + BLEU + COMET — GPT-4o...')


In [9]:
print('Calcul MEDCON + BLEU + COMET — GPT-4o...')
results_gpt = []
for i, doc in enumerate(tqdm(test_docs, desc='MEDCON GPT-4o')):
    r = medcon_grouped(
        doc['text_en'],
        doc['gpt_translation'],
        pairs, automaton_en, automaton_fr
    )
    bleu = sacrebleu.sentence_bleu(doc['gpt_translation'], [doc['translation_fr']]).score
    results_gpt.append({
        'doc_id'           : i,
        'medcon_f1'        : r['f1'],
        'medcon_precision' : r['precision'],
        'medcon_recall'    : r['recall'],
        'bleu'             : bleu,
    })

df_gpt = pd.DataFrame(results_gpt)

# COMET GPT-4o
if USE_COMET:
    comet_data_gpt = [
        {'src': doc['text_en'], 'mt': doc['gpt_translation'], 'ref': doc['translation_fr']}
        for doc in test_docs
    ]
    comet_output_gpt = comet_model.predict(
        comet_data_gpt,
        batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for j, score in enumerate(comet_output_gpt.scores):
        df_gpt.loc[j, 'comet'] = float(score)
    print(f'COMET moyen GPT-4o : {df_gpt["comet"].mean():.3f}')
else:
    df_gpt['comet'] = float('nan')

print('\n' + '=' * 65)
print('COMPARAISON GPT-4o vs RAG (EuroLLM-1.7B + Embeddings, k=3)')
print('=' * 65)
print(f"\n{'Système':<30} {'MEDCON F1':>10} {'BLEU':>8} {'COMET':>8}")
print('-' * 60)
print(f"{'GPT-4o':<30} {df_gpt['medcon_f1'].mean():>10.3f} {df_gpt['bleu'].mean():>8.2f} {df_gpt['comet'].mean():>8.3f}")
print(f"{'RAG (EuroLLM + Embeddings)':<30} {df_rag['medcon_f1'].mean():>10.3f} {df_rag['bleu'].mean():>8.2f} {df_rag['comet'].mean():>8.3f}")

delta_medcon = df_rag['medcon_f1'] - df_gpt['medcon_f1']
delta_bleu   = df_rag['bleu']      - df_gpt['bleu']
delta_comet  = df_rag['comet']     - df_gpt['comet'] if USE_COMET else None

print(f'\nDelta MEDCON F1 (RAG - GPT-4o) : {delta_medcon.mean():+.3f}')
print(f'  Docs où RAG > GPT-4o : {(delta_medcon > 0).sum()}/{len(test_docs)}')
print(f'  Docs où RAG < GPT-4o : {(delta_medcon < 0).sum()}/{len(test_docs)}')
print(f'  Docs égaux           : {(delta_medcon == 0).sum()}/{len(test_docs)}')
print(f'\nDelta BLEU  (RAG - GPT-4o) : {delta_bleu.mean():+.2f}')
if USE_COMET:
    print(f'Delta COMET (RAG - GPT-4o) : {delta_comet.mean():+.3f}')


Calcul MEDCON + BLEU + COMET — GPT-4o...


MEDCON GPT-4o:   0%|          | 0/49 [00:00<?, ?it/s]


COMPARAISON GPT-4o vs RAG (EuroLLM-1.7B + Embeddings, k=3)

Système                         MEDCON F1     BLEU    COMET
------------------------------------------------------------
GPT-4o                              0.398    49.15      nan
RAG (EuroLLM + Embeddings)          0.414    49.58      nan

Delta MEDCON F1 (RAG - GPT-4o) : +0.016
  Docs où RAG > GPT-4o : 1/49
  Docs où RAG < GPT-4o : 1/49
  Docs égaux           : 47/49

Delta BLEU  (RAG - GPT-4o) : +0.42


## 9. Exemples qualitatifs

In [10]:
best_idx  = delta_medcon.idxmax()
worst_idx = delta_medcon.idxmin()

for idx, label in [(best_idx, 'MEILLEURE AMÉLIORATION RAG'), (worst_idx, 'PLUS FORTE DÉGRADATION RAG')]:
    print(f"\n{'#'*80}\n  {label}  (doc #{idx})\n{'#'*80}")
    print(f"\nSOURCE EN :\n{test_docs[idx]['text_en'][:400]}...")
    print(f"\nGPT-4o FR :\n{test_docs[idx]['gpt_translation'][:400]}...")
    print(f"\nRAG FR :\n{rag_translations[idx][:400]}...")
    print(f"\n{'─'*60}")
    print(f"MEDCON F1  GPT-4o={df_gpt.iloc[idx]['medcon_f1']:.3f}  RAG={df_rag.iloc[idx]['medcon_f1']:.3f}  Δ={delta_medcon.iloc[idx]:+.3f}")
    print(f"BLEU       GPT-4o={df_gpt.iloc[idx]['bleu']:.1f}      RAG={df_rag.iloc[idx]['bleu']:.1f}")
    print(f"\nMatchées RAG : {df_rag.iloc[idx]['matched'][:5]}")
    print(f"Manquées RAG : {df_rag.iloc[idx]['missed'][:5]}")


################################################################################
  MEILLEURE AMÉLIORATION RAG  (doc #7)
################################################################################

SOURCE EN :
Melanocytes are located in various parts of the human body, such as the skin and the eye. Their transformation leads to melanoma, an aggressive and deadly neoplasm. Cutaneous and uveal melanomas show different characteristics, including significant differences in genetic alterations, metastatic sites and therapeutic response. In recent decades, great efforts have been made to obtain a more compreh...

GPT-4o FR :
Les mélanocytes sont situés dans diverses parties du corps humain, telles que la peau et l'œil. Leur transformation conduit au mélanome, une néoplasie agressive et mortelle. Les mélanomes cutanés et uvéaux présentent des caractéristiques différentes, y compris des différences significatives dans les altérations génétiques, les sites métastatiques et la réponse théra

## 10. Nettoyage metadata widgets (avant push)

In [11]:
import nbformat

for path in [
    'notebooks/01_medcon_evaluation.ipynb',
    'notebooks/02_dict_comparison.ipynb',
    'notebooks/03_doctor_annotations.ipynb',
    'notebooks/04_rag_embeddings.ipynb',
]:
    if os.path.exists(path):
        nb = nbformat.read(path, as_version=4)
        if 'widgets' in nb.metadata:
            del nb.metadata['widgets']
        nbformat.write(nb, path)
        print(f'Nettoyé : {path}')

Nettoyé : notebooks/01_medcon_evaluation.ipynb
Nettoyé : notebooks/02_dict_comparison.ipynb
Nettoyé : notebooks/03_doctor_annotations.ipynb
